# Bank Marketing – Feature Selection and Prediction (Phases 4–7)

## Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     GridSearchCV, cross_val_predict)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
                             precision_recall_curve, mutual_info_score)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

In [ ]:
LOCAL_PATH = Path('../dataset/bank/bank-full.csv')
DATASET_URL = 'https://raw.githubusercontent.com/MinhTuan2405/bankMarAnalysis/main/dataset/bank/bank-full.csv'
DATA_SOURCE = LOCAL_PATH if LOCAL_PATH.exists() else DATASET_URL
DATASET_SEPARATOR = ';'

TARGET = 'y'
RANDOM_STATE = 42
TEST_SIZE = 0.30
N_SPLITS = 5
N_JOBS = -1

CV = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
SCORING = {'pr_auc': 'average_precision', 'roc_auc': 'roc_auc', 'f1': 'f1',
           'recall': 'recall', 'precision': 'precision', 'accuracy': 'accuracy'}
MAIN_METRIC = 'pr_auc'   # imbalanced data -> PR-AUC is the main metric

## Load data and train/test split

In [ ]:
df = pd.read_csv(DATA_SOURCE, sep=DATASET_SEPARATOR).drop_duplicates().reset_index(drop=True)
df[TARGET] = df[TARGET].map({'yes': 1, 'no': 0}).astype(int)

LEAKAGE_COLS = ['duration']
VALID_FEATURES = [c for c in df.columns if c not in LEAKAGE_COLS + [TARGET]]

X_train, X_test, y_train, y_test = train_test_split(
    df[VALID_FEATURES], df[TARGET], test_size=TEST_SIZE, stratify=df[TARGET], random_state=RANDOM_STATE)

NUM_FEATURES = X_train.select_dtypes('number').columns.tolist()
CAT_FEATURES = X_train.select_dtypes(exclude='number').columns.tolist()
PREVALENCE = y_train.mean()
print(f'Train: {X_train.shape} | Test: {X_test.shape} | y=1 rate (train): {PREVALENCE:.4f}')
print('Numeric    :', NUM_FEATURES)
print('Categorical:', CAT_FEATURES)

# Phase 4. Feature Analysis and Selection (training set only)
## Step 7. Correlation analysis

In [ ]:
pb_rows = []
for c in NUM_FEATURES:
    r, p = stats.pointbiserialr(y_train, X_train[c])
    rho, p_s = stats.spearmanr(X_train[c], y_train)
    pb_rows.append((c, r, p, rho, p_s))
pb_table = (pd.DataFrame(pb_rows, columns=['feature', 'point_biserial_r', 'p_value', 'spearman_rho', 'spearman_p'])
            .set_index('feature').sort_values('point_biserial_r', key=np.abs, ascending=False))
pb_table

In [ ]:
corr = X_train[NUM_FEATURES].join(y_train).corr(method='pearson')
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Pearson correlation – numeric features and y (train)')
plt.show()

In [ ]:
# Strongly correlated numeric pairs (redundant information)
pairs = corr.drop(index=TARGET, columns=TARGET).where(np.triu(np.ones((len(NUM_FEATURES),) * 2, dtype=bool), k=1)).stack()
num_pairs = pairs.rename('pearson_r').to_frame().sort_values('pearson_r', key=np.abs, ascending=False)
num_pairs.head(8)

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    k = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * k)) if k > 0 else 0.0

# Association between categorical features
cv_matrix = pd.DataFrame(
    [[cramers_v(X_train[a], X_train[b]) for b in CAT_FEATURES] for a in CAT_FEATURES],
    index=CAT_FEATURES, columns=CAT_FEATURES)
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cv_matrix, annot=True, fmt='.2f', cmap='Oranges', vmin=0, vmax=1, ax=ax)
ax.set_title("Cramér's V between categorical features (train)")
plt.show()

cat_pairs = (cv_matrix.where(np.triu(np.ones(cv_matrix.shape, dtype=bool), k=1)).stack()
             .rename('cramers_v').to_frame().sort_values('cramers_v', ascending=False))
cat_pairs.head(8)

## Step 8. Statistical tests

In [ ]:
def strength(v):
    return 'very weak' if v < 0.05 else 'weak' if v < 0.1 else 'moderate' if v < 0.3 else 'strong'

chi_rows = []
for c in CAT_FEATURES:
    ct = pd.crosstab(X_train[c], y_train)
    chi2, p, dof, _ = stats.chi2_contingency(ct, correction=False)
    v = cramers_v(X_train[c], y_train)
    chi_rows.append((c, chi2, dof, p, v, strength(v)))
chi_table = (pd.DataFrame(chi_rows, columns=['feature', 'chi2', 'dof', 'p_value', 'cramers_v', 'strength'])
             .set_index('feature').sort_values('cramers_v', ascending=False))
chi_table

In [ ]:
# Welch t-test (2 groups, equivalent to one-way ANOVA) + Mann-Whitney U because of skewness
t_rows = []
for c in NUM_FEATURES:
    a, b = X_train.loc[y_train == 1, c], X_train.loc[y_train == 0, c]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    f, p_f = stats.f_oneway(a, b)
    _, p_mw = stats.mannwhitneyu(a, b)
    d = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    t_rows.append((c, a.mean(), b.mean(), t, p, f, p_f, p_mw, d))
t_table = (pd.DataFrame(t_rows, columns=['feature', 'mean_yes', 'mean_no', 't_stat', 'p_ttest',
                                         'F_anova', 'p_anova', 'p_mannwhitney', 'cohen_d'])
           .set_index('feature').sort_values('cohen_d', key=np.abs, ascending=False))
t_table

In [ ]:
ALPHA = 0.05

## Step 9. Feature importance

In [ ]:
def basic_preprocessor():
    num = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
    cat = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                    ('ohe', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('num', num, make_column_selector(dtype_include='number')),
                              ('cat', cat, make_column_selector(dtype_exclude='number'))],
                             sparse_threshold=0)

# Preliminary model: Random Forest, permutation importance on a validation split taken from train
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)

prelim = Pipeline([('prep', basic_preprocessor()),
                   ('clf', RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                  class_weight='balanced_subsample',
                                                  n_jobs=N_JOBS, random_state=RANDOM_STATE))])
prelim.fit(X_fit, y_fit)
prelim_val_proba = prelim.predict_proba(X_val)[:, 1]
print(f'Preliminary model – validation PR-AUC: {average_precision_score(y_val, prelim_val_proba):.4f} | '
      f'ROC-AUC: {roc_auc_score(y_val, prelim_val_proba):.4f}')

In [ ]:
perm = permutation_importance(prelim, X_val, y_val, scoring='average_precision',
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=N_JOBS)
perm_imp = pd.DataFrame({'perm_mean': perm.importances_mean, 'perm_std': perm.importances_std},
                        index=X_val.columns).sort_values('perm_mean', ascending=False)

# Impurity importance, one-hot columns summed back to the original feature
names = prelim.named_steps['prep'].get_feature_names_out()
rf_imp = pd.Series(prelim.named_steps['clf'].feature_importances_, index=names)
impurity_imp = pd.Series({c: rf_imp[[n for n in names if n == f'num__{c}' or n.startswith(f'cat__{c}_')]].sum()
                          for c in VALID_FEATURES}, name='impurity')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(perm_imp.index[::-1], perm_imp['perm_mean'][::-1], xerr=perm_imp['perm_std'][::-1], color='#4C72B0')
axes[0].set(title='Permutation importance (PR-AUC drop, validation)', xlabel='PR-AUC drop')
imp_sorted = impurity_imp.sort_values()
axes[1].barh(imp_sorted.index, imp_sorted.values, color='#55A868')
axes[1].set(title='Impurity importance (Random Forest, one-hot summed)', xlabel='Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Combined ranking: statistics + model
assoc = pd.concat([pb_table['point_biserial_r'].abs(), chi_table['cramers_v']]).rename('assoc')
p_values = pd.concat([t_table['p_ttest'], chi_table['p_value']]).rename('p_value')
ranking = pd.concat([assoc, p_values, perm_imp, impurity_imp], axis=1).loc[VALID_FEATURES]
ranking['rank_assoc'] = ranking['assoc'].rank(ascending=False)
ranking['rank_perm'] = ranking['perm_mean'].rank(ascending=False)
ranking['rank_impurity'] = ranking['impurity'].rank(ascending=False)
ranking['avg_rank'] = ranking[['rank_assoc', 'rank_perm', 'rank_impurity']].mean(axis=1)

def classify(r):
    if r['perm_mean'] - r['perm_std'] > 0:
        return 'Potential'
    if r['p_value'] < ALPHA:
        return 'Redundant (associated but no extra contribution)'
    return 'Unsuitable (weak on every criterion)'

ranking['verdict'] = ranking.apply(classify, axis=1)
ranking = ranking.sort_values('avg_rank')
leak_row = pd.DataFrame({'verdict': ['Unsuitable (data leakage)']}, index=['duration'])
pd.concat([ranking, leak_row])

## Step 10. Feature set selection
The reduced set is chosen by `AssociationSelector` **inside the Pipeline**, so it is re-learned on every CV fold (no leakage). A feature is dropped only when it is weak on **both** criteria (association coefficient and mutual information); redundant numeric features (|r| > 0.8) are also removed.

In [ ]:
def association_scores(X, y, bins=10):
    y = pd.Series(np.asarray(y), index=X.index)
    rows = {}
    for c in X.columns:
        s = X[c]
        if pd.api.types.is_numeric_dtype(s):
            s = s.fillna(s.min() - 1)
            a = abs(stats.pointbiserialr(y, s)[0]) if s.nunique() > 1 else 0.0
            mi = mutual_info_score(y, pd.qcut(s, q=bins, labels=False, duplicates='drop'))
        else:
            a = cramers_v(s, y)
            mi = mutual_info_score(y, s.astype(str))
        rows[c] = (a, mi)
    return pd.DataFrame.from_dict(rows, orient='index', columns=['assoc', 'mi'])


class AssociationSelector(BaseEstimator, TransformerMixin):
    """Keep a feature if assoc >= assoc_thr OR mi >= mi_thr; drop redundant numeric features (|r| > redundancy_thr)."""

    def __init__(self, assoc_thr=0.05, mi_thr=0.005, redundancy_thr=0.8):
        self.assoc_thr = assoc_thr
        self.mi_thr = mi_thr
        self.redundancy_thr = redundancy_thr

    def fit(self, X, y):
        self.scores_ = association_scores(X, y)
        keep = (self.scores_['assoc'] >= self.assoc_thr) | (self.scores_['mi'] >= self.mi_thr)
        selected = self.scores_.index[keep].tolist()
        num = [c for c in selected if pd.api.types.is_numeric_dtype(X[c])]
        if len(num) > 1:
            r = X[num].corr().abs()
            for i, a in enumerate(num):
                for b in num[i + 1:]:
                    if a in selected and b in selected and r.loc[a, b] > self.redundancy_thr:
                        selected.remove(a if self.scores_.loc[a, 'mi'] < self.scores_.loc[b, 'mi'] else b)
        self.selected_ = selected
        self.dropped_ = [c for c in X.columns if c not in selected]
        return self

    def transform(self, X):
        return X[self.selected_]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.selected_)

In [ ]:
selector_train = AssociationSelector().fit(X_train, y_train)
FULL_SET = VALID_FEATURES
REDUCED_SET = selector_train.selected_
selector_train.scores_.assign(selected=lambda d: d.index.isin(REDUCED_SET)).sort_values('mi', ascending=False)

In [ ]:
print(f'Full set ({len(FULL_SET)}):', FULL_SET)
print(f'Reduced set ({len(REDUCED_SET)}):', REDUCED_SET)
print('Dropped:', selector_train.dropped_)

In [ ]:
# Compare both sets with CV on train (the selector is refit inside each fold)
def prelim_pipe(model, reduced):
    steps = [('select', AssociationSelector())] if reduced else []
    return Pipeline(steps + [('prep', basic_preprocessor()), ('clf', model)])

PRELIM_MODELS = {
    'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                           class_weight='balanced', random_state=RANDOM_STATE),
}

fs_rows = []
for name, model in PRELIM_MODELS.items():
    for fs, reduced in [('full', False), ('reduced', True)]:
        res = cross_validate(prelim_pipe(model, reduced), X_train, y_train, cv=CV, scoring=SCORING, n_jobs=N_JOBS)
        fs_rows.append({'model': name, 'feature_set': fs,
                        **{f'{m}_mean': res[f'test_{m}'].mean() for m in SCORING},
                        f'{MAIN_METRIC}_std': res[f'test_{MAIN_METRIC}'].std()})
fs_compare = pd.DataFrame(fs_rows).set_index(['model', 'feature_set'])
fs_compare

In [ ]:
delta = (fs_compare[f'{MAIN_METRIC}_mean'].xs('reduced', level='feature_set')
         - fs_compare[f'{MAIN_METRIC}_mean'].xs('full', level='feature_set'))
TOLERANCE = 0.005
CHOSEN_SET_NAME = 'reduced' if (delta >= -TOLERANCE).all() else 'full'
CHOSEN_SET = REDUCED_SET if CHOSEN_SET_NAME == 'reduced' else FULL_SET
print('PR-AUC difference (reduced - full):', delta.round(4).to_dict())
print(f'Chosen feature set: {CHOSEN_SET_NAME} ({len(CHOSEN_SET)} features)')

# Phase 5. Data Preparation for Modelling
## Step 11. Feature engineering and encoding

In [ ]:
AGE_BINS = [0, 25, 30, 40, 50, 60, np.inf]
AGE_LABELS = ['<=25', '26-30', '31-40', '41-50', '51-60', '>60']

def add_features(X):
    """Row-wise transforms that learn nothing from data -> safe before CV."""
    X = X.copy()
    X['pdays'] = X['pdays'].replace(-1, np.nan)        # -1 = never contacted -> missing, handled by imputer + indicator
    if 'age' in X:
        X['age_group'] = pd.cut(X['age'], AGE_BINS, labels=AGE_LABELS).astype(str)   # non-linear age effect (EDA)
    return X

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

class ColumnsPresent:
    """Select only the columns present in X (the reduced set may lack some)."""
    def __init__(self, cols):
        self.cols = cols
    def __call__(self, X):
        return [c for c in self.cols if c in X.columns]

SKEWED_COLS = ['balance', 'campaign', 'previous']
PDAYS_COLS = ['pdays']
PLAIN_NUM_COLS = ['age', 'day']
CAT_COLS = CAT_FEATURES + ['age_group']

def make_preprocessor(scale=True):
    scaler = lambda: StandardScaler() if scale else 'passthrough'
    skew = Pipeline([('log', FunctionTransformer(signed_log1p, feature_names_out='one-to-one')), ('scale', scaler())])
    pdays = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', scaler())])
    num = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', scaler())])
    cat = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                    ('ohe', OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('skew', skew, ColumnsPresent(SKEWED_COLS)),
                              ('pdays', pdays, ColumnsPresent(PDAYS_COLS)),
                              ('num', num, ColumnsPresent(PLAIN_NUM_COLS)),
                              ('cat', cat, ColumnsPresent(CAT_COLS))],
                             remainder='drop', sparse_threshold=0)

def make_pipeline(model, feature_set='full', scale=True):
    steps = [('fe', FunctionTransformer(add_features))]
    if feature_set == 'reduced':
        steps.append(('select', AssociationSelector()))
    steps += [('prep', make_preprocessor(scale)), ('clf', model)]
    return Pipeline(steps)

In [ ]:
# Pipeline check: fit on train, transform train/test
demo = make_pipeline(LogisticRegression(max_iter=2000), feature_set=CHOSEN_SET_NAME).fit(X_train, y_train)
prep_demo = demo[:-1]
Xt_train, Xt_test = prep_demo.transform(X_train), prep_demo.transform(X_test)
print('Shape after transform – train:', Xt_train.shape, '| test:', Xt_test.shape)
print('NaN after transform:', int(np.isnan(Xt_train).sum()), int(np.isnan(Xt_test).sum()))
demo

In [ ]:
feature_names_out = prep_demo.named_steps['prep'].get_feature_names_out()
pd.DataFrame(Xt_train[:5], columns=feature_names_out)

# Phase 6. Modelling
## Step 12. Train the classifiers

In [ ]:
MODELS = {
    'Dummy': (DummyClassifier(strategy='most_frequent'), True),
    'LogisticRegression': (LogisticRegression(max_iter=2000, class_weight='balanced'), True),
    'DecisionTree': (DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE), False),
}
FEATURE_SETS = ['full', 'reduced']

def scores(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {'accuracy': accuracy_score(y_true, pred),
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred),
            'f1': f1_score(y_true, pred),
            'roc_auc': roc_auc_score(y_true, proba),
            'pr_auc': average_precision_score(y_true, proba)}

base_rows, base_models = [], {}
for name, (model, scale) in MODELS.items():
    for fs in FEATURE_SETS:
        pipe = make_pipeline(model, fs, scale)
        cv_res = cross_validate(pipe, X_train, y_train, cv=CV, scoring=SCORING, n_jobs=N_JOBS)
        pipe.fit(X_train, y_train)
        base_models[(name, fs)] = pipe
        tr = scores(y_train, pipe.predict_proba(X_train)[:, 1])
        base_rows.append({'model': name, 'feature_set': fs,
                          'train_pr_auc': tr['pr_auc'], 'train_f1': tr['f1'],
                          **{f'cv_{m}': cv_res[f'test_{m}'].mean() for m in SCORING}})
base_results = pd.DataFrame(base_rows).set_index(['model', 'feature_set'])
base_results

## Step 13. Cross-validation and tuning (GridSearchCV + Stratified K-Fold)

In [ ]:
PARAM_GRIDS = {
    'LogisticRegression': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__class_weight': [None, 'balanced'],
    },
    'DecisionTree': {
        'clf__max_depth': [3, 5, 7, 10, None],
        'clf__min_samples_leaf': [1, 20, 50, 100],
        'clf__criterion': ['gini', 'entropy'],
        'clf__class_weight': [None, 'balanced'],
    },
}

searches = {}
for name, grid in PARAM_GRIDS.items():
    model, scale = MODELS[name]
    for fs in FEATURE_SETS:
        gs = GridSearchCV(make_pipeline(model, fs, scale), grid, scoring=SCORING, refit=MAIN_METRIC,
                          cv=CV, n_jobs=N_JOBS, return_train_score=True)
        gs.fit(X_train, y_train)
        searches[(name, fs)] = gs
        print(f'{name:<20} {fs:<8} best {MAIN_METRIC} = {gs.best_score_:.4f} | {gs.best_params_}')

In [ ]:
cv_rows = []
for (name, fs), gs in searches.items():
    i = gs.best_index_
    r = gs.cv_results_
    cv_rows.append({'model': name, 'feature_set': fs,
                    'best_params': {k.replace('clf__', ''): v for k, v in gs.best_params_.items()},
                    **{f'cv_{m}': r[f'mean_test_{m}'][i] for m in SCORING},
                    'cv_pr_auc_std': r[f'std_test_{MAIN_METRIC}'][i],
                    'train_pr_auc': r[f'mean_train_{MAIN_METRIC}'][i]})
cv_summary = pd.DataFrame(cv_rows).set_index(['model', 'feature_set'])
cv_summary.drop(columns='best_params')

In [ ]:
params_table = cv_summary['best_params'].apply(pd.Series)
params_table

In [ ]:
# Effect of the main hyperparameters
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for fs in FEATURE_SETS:
    r = pd.DataFrame(searches[('LogisticRegression', fs)].cv_results_)
    r = r[r['param_clf__class_weight'].astype(str) == str(searches[('LogisticRegression', fs)].best_params_['clf__class_weight'])]
    axes[0].plot(r['param_clf__C'].astype(float), r[f'mean_test_{MAIN_METRIC}'], marker='o', label=fs)
    r = pd.DataFrame(searches[('DecisionTree', fs)].cv_results_)
    best_depth = (r.groupby(r['param_clf__max_depth'].astype(str))[f'mean_test_{MAIN_METRIC}'].max()
                  .reindex([str(d) for d in PARAM_GRIDS['DecisionTree']['clf__max_depth']]))
    axes[1].plot(best_depth.index, best_depth.values, marker='o', label=fs)
axes[0].set(xscale='log', xlabel='C', ylabel='CV PR-AUC', title='Logistic Regression – C')
axes[1].set(xlabel='max_depth', ylabel='CV PR-AUC (best per depth)', title='Decision Tree – max_depth')
for ax in axes:
    ax.legend(title='feature set')
plt.tight_layout()
plt.show()

In [ ]:
# Model choice: highest CV PR-AUC; prefer the reduced set if it is within TOLERANCE (simpler model)
best_key = cv_summary['cv_pr_auc'].idxmax()
if best_key[1] == 'full':
    alt = (best_key[0], 'reduced')
    if cv_summary.loc[alt, 'cv_pr_auc'] >= cv_summary.loc[best_key, 'cv_pr_auc'] - TOLERANCE:
        best_key = alt
FINAL_NAME, FINAL_SET = best_key
final_model = searches[best_key].best_estimator_
print(f'Selected model: {FINAL_NAME} | feature set: {FINAL_SET} | params: {searches[best_key].best_params_}')

In [ ]:
# Threshold that maximises F1 on out-of-fold predictions on train (test not used)
oof_proba = cross_val_predict(final_model, X_train, y_train, cv=CV, method='predict_proba', n_jobs=N_JOBS)[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
BEST_THRESHOLD = float(thr[np.argmax(f1s)])
print(f'Best F1 threshold (OOF train): {BEST_THRESHOLD:.3f} | F1 = {f1s.max():.4f} '
      f'(threshold 0.5: F1 = {f1_score(y_train, (oof_proba >= 0.5).astype(int)):.4f})')

# Phase 7. Model Evaluation
## Step 14. Test set evaluation

In [ ]:
eval_models = {('Dummy', 'full'): base_models[('Dummy', 'full')]}
eval_models.update({k: gs.best_estimator_ for k, gs in searches.items()})

test_rows = []
for (name, fs), m in eval_models.items():
    proba = m.predict_proba(X_test)[:, 1]
    test_rows.append({'model': name, 'feature_set': fs, 'threshold': 0.5, **scores(y_test, proba)})
final_proba = final_model.predict_proba(X_test)[:, 1]
FINAL_TUNED = f'{FINAL_NAME} (tuned threshold)'
test_rows.append({'model': FINAL_TUNED, 'feature_set': FINAL_SET,
                  'threshold': BEST_THRESHOLD, **scores(y_test, final_proba, BEST_THRESHOLD)})
test_results = pd.DataFrame(test_rows).set_index(['model', 'feature_set'])
test_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for (name, fs), m in eval_models.items():
    if name == 'Dummy':
        continue
    RocCurveDisplay.from_estimator(m, X_test, y_test, ax=axes[0], name=f'{name} ({fs})')
    PrecisionRecallDisplay.from_estimator(m, X_test, y_test, ax=axes[1], name=f'{name} ({fs})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1, label=f'Baseline ({y_test.mean():.3f})')
axes[0].set_title('ROC curve – test')
axes[1].set_title('Precision-Recall curve – test')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
pred_05 = (final_proba >= 0.5).astype(int)
pred_opt = (final_proba >= BEST_THRESHOLD).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, pred, title in [(axes[0], pred_05, 'threshold 0.5'), (axes[1], pred_opt, f'threshold {BEST_THRESHOLD:.3f}')]:
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['no', 'yes']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{FINAL_NAME} ({FINAL_SET}) – {title}')
plt.tight_layout()
plt.show()

In [ ]:
# False Positive / False Negative analysis (tuned threshold)
def outcome_label(t, p):
    return np.select([(t == 1) & (p == 1), (t == 0) & (p == 0), (t == 0) & (p == 1), (t == 1) & (p == 0)],
                     ['TP', 'TN', 'FP', 'FN'], default='NA')

err_df = X_test.copy()
err_df['y'] = y_test.values
err_df['proba'] = final_proba
err_df['outcome'] = outcome_label(y_test.values, pred_opt)
tn, fp, fn, tp = confusion_matrix(y_test, pred_opt).ravel()
err_summary = pd.Series({'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
                         'Recall (TPR)': tp / (tp + fn), 'FNR': fn / (tp + fn),
                         'FPR': fp / (fp + tn), 'Precision': tp / (tp + fp)}, name='value').to_frame()
err_summary

In [ ]:
# Error profiles: share of each group within TP / FN / FP / TN
PROFILE_COLS = ['poutcome', 'contact', 'housing', 'month']
profile = pd.concat({c: pd.crosstab(err_df[c], err_df['outcome'], normalize='columns')
                        .reindex(columns=['TP', 'FN', 'FP', 'TN'], fill_value=0)
                     for c in PROFILE_COLS})
display(profile.style.format('{:.1%}').background_gradient(cmap='Oranges', axis=1))
err_df.groupby('outcome')[['age', 'balance', 'campaign', 'previous', 'proba']].median()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=err_df, x='proba', hue='y', bins=50, stat='density', common_norm=False, element='step', ax=ax)
ax.axvline(BEST_THRESHOLD, color='r', ls='--', label=f'threshold {BEST_THRESHOLD:.3f}')
ax.set(title='Predicted probability distribution (test)', xlabel='P(y = 1)')
ax.legend()
plt.show()

In [ ]:
# Ability to find real subscribers: share captured when calling the top k% scored clients
order = np.argsort(-final_proba)
y_sorted = y_test.values[order]
gain = pd.DataFrame({'top_%': [10, 20, 30, 40, 50]})
gain['clients_called'] = (gain['top_%'] / 100 * len(y_sorted)).astype(int)
gain['%_yes_captured'] = [y_sorted[:k].sum() / y_sorted.sum() for k in gain['clients_called']]
gain['yes_rate_in_group'] = [y_sorted[:k].mean() for k in gain['clients_called']]
gain['lift'] = gain['yes_rate_in_group'] / y_test.mean()
gain

## Step 15. Comparison and analysis

In [ ]:
# Before / after feature selection on test (threshold 0.5)
fs_test = test_results.drop(index=[('Dummy', 'full'), (FINAL_TUNED, FINAL_SET)])
fs_test_pivot = fs_test[['pr_auc', 'roc_auc', 'f1', 'recall', 'precision']].unstack('feature_set')
fs_test_pivot

In [ ]:
# Overfitting / underfitting check: train vs CV vs test (PR-AUC)
fit_rows = []
for (name, fs), gs in searches.items():
    m = gs.best_estimator_
    fit_rows.append({'model': name, 'feature_set': fs,
                     'train_pr_auc': average_precision_score(y_train, m.predict_proba(X_train)[:, 1]),
                     'cv_pr_auc': cv_summary.loc[(name, fs), 'cv_pr_auc'],
                     'test_pr_auc': test_results.loc[(name, fs), 'pr_auc']})
fit_rows.append({'model': 'DecisionTree (default)', 'feature_set': 'full',
                 'train_pr_auc': base_results.loc[('DecisionTree', 'full'), 'train_pr_auc'],
                 'cv_pr_auc': base_results.loc[('DecisionTree', 'full'), 'cv_pr_auc'],
                 'test_pr_auc': average_precision_score(
                     y_test, base_models[('DecisionTree', 'full')].predict_proba(X_test)[:, 1])})
fit_check = pd.DataFrame(fit_rows).set_index(['model', 'feature_set'])
fit_check['gap_train_cv'] = fit_check['train_pr_auc'] - fit_check['cv_pr_auc']
fit_check['gap_cv_test'] = fit_check['cv_pr_auc'] - fit_check['test_pr_auc']
fit_check['assessment'] = np.select(
    [fit_check['gap_train_cv'] > 0.10, fit_check['gap_train_cv'] > 0.03, fit_check['train_pr_auc'] < 2 * PREVALENCE],
    ['Clear overfitting', 'Slight overfitting', 'Underfitting'], 'Balanced')
fit_check

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
fit_check[['train_pr_auc', 'cv_pr_auc', 'test_pr_auc']].plot.bar(ax=ax, rot=20)
ax.axhline(PREVALENCE, color='k', ls='--', lw=1, label='Dummy')
ax.set(title='PR-AUC: train vs CV vs test', ylabel='PR-AUC', xlabel='')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature roles – Logistic Regression (coefficients / odds ratios)
lr_model = searches[('LogisticRegression', FINAL_SET)].best_estimator_
lr_coef = pd.Series(lr_model.named_steps['clf'].coef_[0],
                    index=lr_model.named_steps['prep'].get_feature_names_out())
lr_top = lr_coef.reindex(lr_coef.abs().sort_values(ascending=False).index).head(15)
lr_top_df = pd.DataFrame({'coef': lr_top, 'odds_ratio': np.exp(lr_top)})

fig, ax = plt.subplots(figsize=(8, 6))
lr_top[::-1].plot.barh(ax=ax, color=np.where(lr_top[::-1] > 0, '#DD8452', '#4C72B0'))
ax.set(title=f'Top 15 Logistic Regression coefficients ({FINAL_SET})', xlabel='Coefficient (log-odds)')
plt.tight_layout()
plt.show()
lr_top_df

In [ ]:
# Feature roles – Decision Tree
dt_model = searches[('DecisionTree', FINAL_SET)].best_estimator_
dt_names = list(dt_model.named_steps['prep'].get_feature_names_out())
dt_imp = pd.Series(dt_model.named_steps['clf'].feature_importances_, index=dt_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(dt_model.named_steps['clf'], max_depth=3, feature_names=dt_names, class_names=['no', 'yes'],
          filled=True, rounded=True, fontsize=8, ax=ax, impurity=False, proportion=True)
ax.set_title(f'Decision Tree ({FINAL_SET}) – top 3 levels')
plt.show()
dt_imp.head(15).to_frame('importance')

In [ ]:
# Permutation importance of the final model on test (interpretation only, not used for selection)
final_perm = permutation_importance(final_model, X_test, y_test, scoring='average_precision',
                                    n_repeats=10, random_state=RANDOM_STATE, n_jobs=N_JOBS)
final_perm_imp = pd.DataFrame({'mean': final_perm.importances_mean, 'std': final_perm.importances_std},
                              index=X_test.columns).sort_values('mean', ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(final_perm_imp.index[::-1], final_perm_imp['mean'][::-1], xerr=final_perm_imp['std'][::-1], color='#55A868')
ax.set(title=f'Permutation importance – {FINAL_NAME} ({FINAL_SET}) on test', xlabel='PR-AUC drop')
plt.tight_layout()
plt.show()

In [ ]:
# Overall model comparison
CV_COLS = ['cv_pr_auc', 'cv_roc_auc', 'cv_f1', 'cv_recall']
TEST_COLS = ['pr_auc', 'roc_auc', 'f1', 'recall', 'precision', 'accuracy']
comparison = (test_results[['threshold'] + TEST_COLS].add_prefix('test_')
              .join(cv_summary[CV_COLS], how='left'))
comparison = comparison[CV_COLS + ['test_threshold'] + [f'test_{c}' for c in TEST_COLS]]
comparison